In [1]:
import sys
!{sys.executable} -m pip install pymongo


In [2]:
import pandas as pd
from pymongo import MongoClient

# --- Configuration des fichiers ---
FILES = {
    "caracts": "caracts_nettoye.csv",
    "lieux": "lieux_nettoye.csv",
    "usagers": "usagers_nettoye.csv",
    "vehicules": "vehicules_nettoye.csv",
}

# --- Étape de nettoyage des données pour assurer des jointures correctes ---
def clean_id_columns(df, columns):
    """Nettoie les colonnes d'ID en retirant les espaces et en les convertissant en string."""
    for col in columns:
        if col in df.columns:
            # Remplace les espaces insécables et les espaces normaux
            df[col] = df[col].astype(str).str.replace(r'[\s\xa0]', '', regex=True)
    return df

# Charger les DataFrames
df_caracts = pd.read_csv(FILES["caracts"], sep=',')
df_lieux = pd.read_csv(FILES["lieux"], sep=',')
df_usagers = pd.read_csv(FILES["usagers"], sep=',')
df_vehicules = pd.read_csv(FILES["vehicules"], sep=',')

# Nettoyage des clés de jointure
df_caracts = clean_id_columns(df_caracts, ['num_acc'])
df_lieux = clean_id_columns(df_lieux, ['num_acc'])
df_usagers = clean_id_columns(df_usagers, ['num_acc', 'id_usager', 'id_vehicule'])
df_vehicules = clean_id_columns(df_vehicules, ['num_acc', 'id_vehicule'])

# Renommer la colonne 'num_acc' en '_id' pour l'utiliser comme clé primaire MongoDB
df_caracts = df_caracts.rename(columns={'num_acc': '_id'})

In [3]:
# --- 2. Structuration Usagers -> Véhicules ---
# Supprimer les colonnes déjà présentes dans vehicules pour éviter la redondance
usagers_to_merge = df_usagers.drop(columns=['num_veh', 'num_acc'])

# Grouper les usagers par véhicule
df_usagers_grouped = usagers_to_merge.groupby('id_vehicule').apply(
    lambda x: x.drop('id_vehicule', axis=1).to_dict('records')
).reset_index(name='usagers')

# Joindre les usagers groupés aux véhicules
df_vehicules_with_usagers = df_vehicules.merge(
    df_usagers_grouped, 
    on='id_vehicule', 
    how='left'
)

# --- 3. Structuration Lieux et Véhicules/Usagers -> Accidents ---
# Grouper les véhicules (avec usagers) par accident
df_vehicules_grouped = df_vehicules_with_usagers.groupby('num_acc').apply(
    lambda x: x.drop('num_acc', axis=1).to_dict('records')
).reset_index(name='vehicules')

# Grouper les lieux par accident
df_lieux_grouped = df_lieux.groupby('num_acc').apply(
    lambda x: x.drop('num_acc', axis=1).to_dict('records')
).reset_index(name='lieux')

# --- 4. Fusion Finale dans la table des Caractéristiques ---
# Joindre les véhicules groupés à la table principale (caractéristiques)
final_df = df_caracts.merge(
    df_vehicules_grouped.rename(columns={'num_acc': '_id'}),
    on='_id',
    how='left'
)

# Joindre les lieux groupés à la table principale
final_df = final_df.merge(
    df_lieux_grouped.rename(columns={'num_acc': '_id'}),
    on='_id',
    how='left'
)

# Remplacer les NaN (pour les accidents sans données de véhicules/lieux) par des listes vides
final_df['vehicules'] = final_df['vehicules'].apply(lambda x: x if isinstance(x, list) else [])
final_df['lieux'] = final_df['lieux'].apply(lambda x: x if isinstance(x, list) else [])

# Convertir le DataFrame final en une liste de dictionnaires (documents JSON)
documents = final_df.to_dict('records')

# Afficher les 2 premiers documents pour vérification
print("Exemple des 2 premiers documents structurés pour MongoDB :")
print(documents[0])
print(documents[1])

/var/folders/8y/7gzgf3tj7ysf_jxytwqg58qc0000gn/T/ipykernel_76387/3338573830.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_usagers_grouped = usagers_to_merge.groupby('id_vehicule').apply(
/var/folders/8y/7gzgf3tj7ysf_jxytwqg58qc0000gn/T/ipykernel_76387/3338573830.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_vehicules_grouped = df_vehicules_with_usagers.groupby('num_acc').apply(
/var/folders/8y/7gzg

Exemple des 2 premiers documents structurés pour MongoDB :
{'_id': '202300000001', 'jour': 7, 'mois': 5, 'an': 2023, 'hrmn': '2025-12-05 06:00:00', 'lum': 1, 'dep': '75', 'com': '75101', 'agg': 2, 'int': 4, 'atm': 2, 'col': 7, 'adr': 'Rue De Rivoli', 'lat': 48.866386, 'long': 2.323471, 'vehicules': [{'id_vehicule': '155680557', 'num_veh': 'A01', 'senc': 1, 'catv': 30, 'obs': 0, 'obsm': 0, 'choc': 5, 'manv': 1, 'motor': 1, 'occutc': 0.0, 'usagers': [{'id_usager': '203851184', 'place': 1, 'catu': 1, 'grav': 4, 'sexe': 1, 'an_nais': 1978.0, 'trajet': 5, 'secu1': 2, 'secu2': 0, 'secu3': -1, 'locp': -1, 'actp': '-1', 'etatp': -1, 'age': 45.0}]}], 'lieux': [{'catr': 4, 'voie': 'Rue De Rivoli', 'v1': 0, 'circ': 1, 'nbv': '2', 'vosp': 0, 'prof': 1, 'pr': '-1', 'pr1': '-1', 'plan': 1, 'larrout': '-1', 'surf': 2, 'infra': 0, 'situ': 1, 'vma': 30}, {'catr': 4, 'voie': 'Rue Saint Florentin', 'v1': 0, 'circ': 1, 'nbv': '1', 'vosp': 0, 'prof': 1, 'pr': '-1', 'pr1': '-1', 'plan': 1, 'larrout': '-1', 

In [ ]:
# --- Configuration de la base de données MongoDB ---import osfrom dotenv import load_dotenvfrom pymongo import MongoClientfrom pymongo.server_api import ServerApi# Charger MONGODB_URI depuis le fichier .env (jamais de credentials en dur)load_dotenv()MONGODB_URI = os.getenv('MONGODB_URI')if not MONGODB_URI:    raise RuntimeError("La variable d'environnement MONGODB_URI est manquante (fichier .env)")DATABASE_NAME = "db_accidents_corporels"COLLECTION_NAME = "accidents"# --- Connexion et Insertion ---try:    client = MongoClient(        MONGODB_URI,        server_api=ServerApi("1"),        connectTimeoutMS=30000,        serverSelectionTimeoutMS=30000,        socketTimeoutMS=120000    )    db = client[DATABASE_NAME]    collection = db[COLLECTION_NAME]    # Supprimer les documents existants dans la collection pour une insertion propre    collection.delete_many({})    # Insérer les documents par lots pour éviter les timeouts    BATCH_SIZE = 500    inserted_count = 0    for i in range(0, len(documents), BATCH_SIZE):        batch = documents[i:i + BATCH_SIZE]        result = collection.insert_many(batch)        inserted_count += len(result.inserted_ids)        print(f"  Lot {i // BATCH_SIZE + 1}: {len(result.inserted_ids)} documents insérés")    print("--- Opération MongoDB terminée ---")    print(f"Base de données cible : {DATABASE_NAME}")    print(f"Collection cible : {COLLECTION_NAME}")    print(f"Nombre total de documents insérés : {inserted_count}")    # Fermer la connexion    client.close()except Exception as e:    print(f"Une erreur est survenue lors de l'insertion dans MongoDB : {e}")